# GUS01C: GeoTERYT Database v3.1

This notebook builds and demonstrates the GeoTERYT database system for Polish administrative divisions.

## Overview
- Build database from harmonized TERYT data (from GUS01A)
- Load geometries from shapefiles (from GUS01B) - **storage only**
- Search and query functionality
- Export and save complete database

**NEW in v3.1**: Geometry assignment is now handled separately in GUS01D.
- `load_geometries()` now only stores geometry data
- Actual assignment uses `assign_geometries()`, `assign_missing_geometries()`, etc.
- See GUS01D for geometry methods demonstration

In [1]:
import os
import sys
import pandas as pd
from pathlib import Path
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import importlib
import gc
from pympler import asizeof

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
            return p
    return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'
geospatial_root = repo_root.parent.parent / 'Data' / 'Geospatial'
geometry_root = repo_root.parent.parent / 'Data' / 'Geospatial' / 'geometry'
gus_root = Path(os.getcwd()).parent.parent.parent.parent / "Data" / "GUS"

os.chdir(repo_root)
if str(repo_root / 'Code' / 'tools') not in sys.path:
    sys.path.append(str(repo_root / 'Code' / 'tools'))

# Import local toolkit (with reload for development)
import local_utility_functions as luf
import geoTERYT_db as gtdb
importlib.reload(gtdb)

print(f"Repo root: {repo_root}")
print(f"GUS root: {gus_root}")
print(f"Geometry root: {geometry_root}")
print(f"geoTERYT_db version: {gtdb.__doc__.split(chr(10))[0] if gtdb.__doc__ else 'unknown'}")

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS
Geometry root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geometry
geoTERYT_db version: 


## 1. Load Harmonized TERYT Data

Load the mega DataFrame created in GUS01A notebook.

In [2]:
# Load the saved mega_df from GUS01A/GUS01B
mega_df = pd.read_csv(geospatial_root / "teryt_df.csv", encoding="utf-8")

# Ensure proper formatting
v_tostring = np.vectorize(lambda x: str(x))
v_zero_pad = np.vectorize(lambda x, n: str(x).zfill(n))
mega_df['id'] = v_zero_pad(v_tostring(mega_df['id']), 7)

print(f"Loaded mega_df: {len(mega_df)} rows")
print(f"Years: {mega_df['year'].min()} - {mega_df['year'].max()}")
print(f"Columns: {list(mega_df.columns)}")
mega_df.head()

Loaded mega_df: 107372 rows
Years: 1999 - 2024
Columns: ['year', 'WOJ', 'POW', 'GMI', 'RODZ', 'id', 'NAZWA', 'NAZWA_DOD', 'level', 'kind', 'STAN_NA', 'if_changed', 'when_changed', 'notes', 'historical_codes', 'code_by_year']


,year,WOJ,POW,GMI,RODZ,id,NAZWA,NAZWA_DOD,level,kind,STAN_NA,if_changed,when_changed,notes,historical_codes,code_by_year
0,1999,2,0,0,0,0200000,DOLNOŚLĄSKIE,województwo,2,NaN,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}",['0200000'],"{1999: '0200000', 2000: '0200000', 2001: '0200..."
1,1999,2,1,0,0,0201000,bolesławiecki,powiat,5,NaN,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}",['0201000'],"{1999: '0201000', 2000: '0201000', 2001: '0201..."
2,1999,2,1,1,1,0201011,Bolesławiec,gmina miejska,6,urban,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}",['0201011'],"{1999: '0201011', 2000: '0201011', 2001: '0201..."
3,1999,2,1,2,2,0201022,Bolesławiec,gmina wiejska,6,rural,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}",['0201022'],"{1999: '0201022', 2000: '0201022', 2001: '0201..."
4,1999,2,1,3,2,0201032,Gromadka,gmina wiejska,6,rural,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}",['0201032'],"{1999: '0201032', 2000: '0201032', 2001: '0201..."


## 2. Load Geometry Files

In [3]:
import re

# Scan for geometry files
geometry_files = [f for f in os.listdir(geometry_root) if not f.startswith('.')]
geometry_paths = [geometry_root / f for f in geometry_files]

shape_files = []
for i in range(len(geometry_paths)):
    if geometry_paths[i].is_dir():
        dir_contents = os.listdir(geometry_paths[i])
        for file in dir_contents:
            if file.endswith('.shp'):
                shape_files.append(geometry_paths[i] / file)

shape_files.sort()
# Filter to gmina-level files
shape_files = [f for f in shape_files if 'Obszary' in f.name or 'gmin' in f.name]

print("Found shapefiles:")
for f in shape_files:
    print(f"  {f.parent.name}/{f.name}")

Found shapefiles:
  PRG_jednostki_administracyjne_2005/Obszary.shp
  PRG_jednostki_administracyjne_2006/Obszary.shp
  PRG_jednostki_administracyjne_2007/Obszary.shp
  PRG_jednostki_administracyjne_2008/Obszary.shp
  PRG_jednostki_administracyjne_2009/Obszary.shp
  PRG_jednostki_administracyjne_2010/Obszary.shp
  PRG_jednostki_administracyjne_2011/Obszary.shp
  PRG_jednostki_administracyjne_2012/Obszary_1992_region.shp
  PRG_jednostki_administracyjne_2015/gminy.shp
  PRG_jednostki_administracyjne_2016/gminy.shp
  PRG_jednostki_administracyjne_2017/gminy.shp
  PRG_jednostki_administracyjne_2018/gminy.shp
  PRG_jednostki_administracyjne_2021/A03_Granice_gmin.shp
  PRG_jednostki_administracyjne_2022/A03_Granice_gmin.shp
  PRG_jednostki_administracyjne_2023/A03_Granice_gmin.shp


In [4]:
# Load all shapefiles into a dictionary
gdf_list = {}
for file in shape_files:
    key = str(file.parent)[-4:] + "_" + str(file.stem)
    try:
        gdf = gpd.read_file(file)
        gdf_list[key] = gdf
        print(f"Loaded {key}: {len(gdf)} features")
    except Exception as e:
        print(f"Error loading {key}: {e}")

print(f"\nTotal loaded: {len(gdf_list)} GeoDataFrames")

Loaded 2005_Obszary: 3175 features
Loaded 2006_Obszary: 3183 features
Loaded 2007_Obszary: 3183 features
Loaded 2008_Obszary: 3185 features
Loaded 2009_Obszary: 3190 features
Loaded 2010_Obszary: 3203 features
Loaded 2011_Obszary: 3209 features
Loaded 2012_Obszary_1992_region: 3209 features
Loaded 2015_gminy: 2481 features
Loaded 2016_gminy: 2481 features
Loaded 2017_gminy: 2478 features
Loaded 2018_gminy: 2478 features
Loaded 2021_A03_Granice_gmin: 2477 features
Loaded 2022_A03_Granice_gmin: 2477 features
Loaded 2023_A03_Granice_gmin: 2477 features

Total loaded: 15 GeoDataFrames


In [5]:
# Prepare GeoDataFrames: standardize teryt column and CRS
for key in gdf_list:
    print(f"Processing {key}...")
    
    # Extract year from key
    match = re.search(r'\d{4}', key)
    if not match:
        continue
    year = int(match.group(0))
    
    # Standardize column names to lowercase
    gdf_list[key].columns = [col.lower() for col in gdf_list[key].columns]
    
    # Convert CRS to EPSG:2180
    if gdf_list[key].crs is None:
        gdf_list[key] = gdf_list[key].set_crs('EPSG:2180')
    else:
        gdf_list[key] = gdf_list[key].to_crs('EPSG:2180')
    
    # Standardize TERYT column
    if year > 2012:
        gdf_list[key]['teryt'] = gdf_list[key]['jpt_kod_je']
    else:
        # Older files use 'obszar' with NUTS-like codes
        v_NCtT = np.vectorize(luf.nuts_code_to_teryt)
        v_tostring = np.vectorize(lambda x: str(x))
        gdf_list[key]['teryt'] = v_NCtT(v_tostring(gdf_list[key]['obszar'].values))
    
    # Sort by teryt
    gdf_list[key] = gdf_list[key].sort_values(by='teryt').reset_index(drop=True)
    
print("\nAll GeoDataFrames processed!")

Processing 2005_Obszary...
Processing 2006_Obszary...
Processing 2007_Obszary...
Processing 2008_Obszary...
Processing 2009_Obszary...
Processing 2010_Obszary...
Processing 2011_Obszary...
Processing 2012_Obszary_1992_region...
Processing 2015_gminy...
Processing 2016_gminy...
Processing 2017_gminy...
Processing 2018_gminy...
Processing 2021_A03_Granice_gmin...
Processing 2022_A03_Granice_gmin...
Processing 2023_A03_Granice_gmin...

All GeoDataFrames processed!


In [6]:
# For older geometries (<=2012), dissolve sub-units (RODZ 4,5) into main gmina
for key in gdf_list:
    match = re.search(r'\d{4}', key)
    if not match:
        continue
    year = int(match.group(0))
    
    if year <= 2012:
        print(f"Dissolving sub-units for {key}...")
        
        # Fix invalid geometries
        invalid_mask = ~gdf_list[key].is_valid
        if invalid_mask.any():
            print(f"  Fixing {invalid_mask.sum()} invalid geometries...")
            gdf_list[key].loc[invalid_mask, 'geometry'] = gdf_list[key].loc[invalid_mask, 'geometry'].buffer(0)
        
        # Create short teryt (replace last digit with '0')
        gdf_list[key]['teryt_short'] = gdf_list[key]['teryt'].str[:-1] + '0'
        len_before = len(gdf_list[key])
        gdf_list[key] = gdf_list[key].dissolve(by='teryt_short', as_index=False)
        gdf_list[key]['teryt'] = gdf_list[key]['teryt_short']
        gdf_list[key] = gdf_list[key].drop(columns=['teryt_short'])
        gdf_list[key] = gdf_list[key].sort_values(by='teryt').reset_index(drop=True)
        print(f"  Dissolved {len_before - len(gdf_list[key])} features. Current total: {len(gdf_list[key])} units.")
        
        '''
        print(f"  Result: {len(gdf_list[key])} features")
        if year <= 2002:
            print(f"  Handling Warsaw-case for {key}...")
            # For Warsaw, we need to dissolve the 18 districts into one unit
            warsaw_mask = gdf_list[key]['teryt'].apply(lambda x: (str(x).startswith('1431') and str(x)[5] in ['1', '2', '3', '4', '5', '6', '7', '8', '9'] and str(x)[6] == "1") or str(x) == '1431201')
            print(len(gdf_list[key][warsaw_mask]), "features identified as Warsaw districts.")
            if warsaw_mask.any():
                print(f"   Dissolving Warsaw districts into one unit...")
                gdf_list[key].loc[warsaw_mask, 'teryt'] = '1431001'
                len_before = len(gdf_list[key])
                gdf_list[key] = gdf_list[key].dissolve(by='teryt', as_index=False)
                gdf_list[key] = gdf_list[key].sort_values(by='teryt').reset_index(drop=True)
                print(f"   Dissolved {len_before - len(gdf_list[key])} features. Current total: {len(gdf_list[key])} units.")
        '''
        
        print(f"Dissolving sub-types for {key}...")
        gdf_list[key]['teryt_short'] = gdf_list[key]['teryt'].apply(lambda x: x[:-2] + '3' if x[-2] in ['4', '5', '6', '7', '8', '9'] else x[:-1])
        len_before = len(gdf_list[key])
        gdf_list[key] = gdf_list[key].dissolve(by='teryt_short', as_index=False)
        gdf_list[key]['teryt'] = gdf_list[key]['teryt_short']
        gdf_list[key] = gdf_list[key].drop(columns=['teryt_short'])
        gdf_list[key] = gdf_list[key].sort_values(by='teryt').reset_index(drop=True)
        print(f"  Dissolved {len_before - len(gdf_list[key])} features. Current total: {len(gdf_list[key])} units.")
        
    else:
        print(f"Analzying {key} for invalid geometries...")
        invalid_mask = ~gdf_list[key].is_valid
        if invalid_mask.any():
            print(f" Fixing {invalid_mask.sum()} invalid geometries...")
            gdf_list[key].loc[invalid_mask, 'geometry'] = gdf_list[key].loc[invalid_mask, 'geometry'].buffer(0)

print("\nDissolution complete!")
gc.collect()

Dissolving sub-units for 2005_Obszary...
  Fixing 7 invalid geometries...
  Dissolved 118 features. Current total: 3057 units.
Dissolving sub-types for 2005_Obszary...
  Dissolved 580 features. Current total: 2477 units.
Dissolving sub-units for 2006_Obszary...
  Fixing 8 invalid geometries...
  Dissolved 123 features. Current total: 3060 units.
Dissolving sub-types for 2006_Obszary...
  Dissolved 582 features. Current total: 2478 units.
Dissolving sub-units for 2007_Obszary...
  Fixing 7 invalid geometries...
  Dissolved 121 features. Current total: 3062 units.
Dissolving sub-types for 2007_Obszary...
  Dissolved 584 features. Current total: 2478 units.
Dissolving sub-units for 2008_Obszary...
  Fixing 9 invalid geometries...
  Dissolved 121 features. Current total: 3064 units.
Dissolving sub-types for 2008_Obszary...
  Dissolved 586 features. Current total: 2478 units.
Dissolving sub-units for 2009_Obszary...
  Fixing 7 invalid geometries...
  Dissolved 121 features. Current total: 3

73

## 3. Build the GeoTERYT Database

In [7]:
# Create and build the database
db = gtdb.GeoTERYTDatabase()
db.build_from_harmonized(mega_df, verbose=True)

Building GeoTERYT database v2.0 from harmonized data...
  Pass 1: Collecting unit states by year...
    Found 4560 unique TERYT IDs across all years
  Pass 2: Building records with change detection...
  Pass 2.5: Populating historical codes and code_by_year...
    Records with historical_codes: 4560
    Records with code_by_year: 4560
    Name changes detected: 33
    Kind changes detected: 0
    Level changes detected: 0
    Changes from notes: 1600
  Pass 3: Building indices...

✓ Database built successfully!
  Total records: 4,560
  Year range: 1999 - 2024
  Voivodeships: 16
  Powiats: 382
  Gminas: 4162
  Units with changes: 772
  Units with level changes: 0
  Units with kind changes: 0


In [8]:
# Load shape of Poland
db.load_poland_shape()

# Load geometries (v3.1: STORAGE ONLY, does NOT assign to records)
# Use db.assign_geometries() methods (in GUS01D) for actual assignment
db.load_geometries(gdf_list, teryt_column='teryt', verbose=True)

gc.collect()

Loading Poland shape from None...
  ✓ Poland shape loaded and set for clipping
Loading geometries into database (v3.1 - storage only)...
  (Clipping to Poland boundary enabled)
  Processing 2005_Obszary (year 2005)...
    Stored 2477 geometries for year 2005
  Processing 2006_Obszary (year 2006)...
    Stored 2478 geometries for year 2006
  Processing 2007_Obszary (year 2007)...
    Stored 2478 geometries for year 2007
  Processing 2008_Obszary (year 2008)...
    Stored 2478 geometries for year 2008
  Processing 2009_Obszary (year 2009)...
    Stored 2478 geometries for year 2009
  Processing 2010_Obszary (year 2010)...
    Stored 2479 geometries for year 2010
  Processing 2011_Obszary (year 2011)...
    Stored 2479 geometries for year 2011
  Processing 2012_Obszary_1992_region (year 2012)...
    Stored 2479 geometries for year 2012
  Processing 2015_gminy (year 2015)...
    Stored 2481 geometries for year 2015
  Processing 2016_gminy (year 2016)...
    Stored 2481 geometries for year 

0

In [9]:
# Save the complete database (with geometries) using pickle
# If db was created before module reload, attach the new methods
if not hasattr(db, 'save_complete'):
    import types
    db.save_complete = types.MethodType(gtdb.GeoTERYTDatabase.save_complete, db)
    print("ℹ Added save_complete method from reloaded module")

complete_db_path = geospatial_root / "geoteryt_complete.pkl"
db.save_complete(complete_db_path)
print(f"\n✓ Complete database saved to: {complete_db_path}")

# Clear memory from large objects
del gdf_list
gc.collect()

Saving complete database to /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete.pkl...
  ✓ Saved 4560 records
  ✓ File size: 1059.6 MB
  ✓ Path: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete.pkl

✓ Complete database saved to: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete.pkl


0

In [10]:
# Print database summary
db.print_summary()

GeoTERYT Database Summary (v3.0)
Total records:           4,560
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
------------------------------------------------------------
Change tracking:
  Records with changes:      772
  Records with level changes: 0
  Records with kind changes:  0
------------------------------------------------------------
Geometry:
  Records with geometry: 0
  Geometry years:        [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  Poland boundary set:   True


In [11]:
df = gtdb.as_df(db.get_divisions_by_year(2017, 6))
display(df)

gc.collect()

,teryt_id,name,name_dod,level,kind,woj,pow,gmi,rodz,years_valid,...,changes,has_changes,n_changes,has_geometry,geometry_year,geometry_notes,old_woj,old_woj_id,historical_codes,code_by_year
0,0807012,Krzeszyce,gmina wiejska,6,rural,08,07,01,2,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,[],False,0,False,None,None,None,None,[0807012],"{1999: '0807012', 2000: '0807012', 2001: '0807..."
1,2401052,Mierzęcice,gmina wiejska,6,rural,24,01,05,2,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,[],False,0,False,None,None,None,None,[2401052],"{1999: '2401052', 2000: '2401052', 2001: '2401..."
2,3028053,Skoki,gmina miejsko-wiejska,6,urban-rural,30,28,05,3,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,[],False,0,False,None,None,None,None,[3028053],"{1999: '3028053', 2000: '3028053', 2001: '3028..."
3,1818062,Zaleszany,gmina wiejska,6,rural,18,18,06,2,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,[],False,0,False,None,None,None,None,[1818062],"{1999: '1818062', 2000: '1818062', 2001: '1818..."
4,3005043,Rakoniewice,gmina miejsko-wiejska,6,urban-rural,30,05,04,3,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,[],False,0,False,None,None,None,None,[3005043],"{1999: '3005043', 2000: '3005043', 2001: '3005..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2474,2415092,Mszana,gmina wiejska,6,rural,24,15,09,2,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,[],False,0,False,None,None,None,None,[2415092],"{1999: '2415092', 2000: '2415092', 2001: '2415..."
2475,0619052,Urszulin,gmina wiejska,6,rural,06,19,05,2,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,[],False,0,False,None,None,None,None,[0619052],"{1999: '0619052', 2000: '0619052', 2001: '0619..."
2476,2411033,Krzanowice,gmina miejsko-wiejska,6,urban-rural,24,11,03,3,"[2001, 2002, 2003, 2004, 2005, 2006, 2007, 200...",...,"[{'year': 2001, 'type': 'reform', 'description...",True,2,False,None,None,None,None,"[2411032, 2411033]","{1999: '2411032', 2000: '2411032', 2001: '2411..."
2477,1212042,Klucze,gmina wiejska,6,rural,12,12,04,2,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,[],False,0,False,None,None,None,None,[1212042],"{1999: '1212042', 2000: '1212042', 2001: '1212..."


0

In [12]:
import os
import sys
import pandas as pd
from pathlib import Path
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import importlib
from pympler import asizeof
import gc

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
            return p
    return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'
geospatial_root = repo_root.parent.parent / 'Data' / 'Geospatial'
geometry_root = repo_root.parent.parent / 'Data' / 'Geospatial' / 'geometry'
gus_root = Path(os.getcwd()).parent.parent.parent.parent / "Data" / "GUS"

os.chdir(repo_root)
if str(repo_root / 'Code' / 'tools') not in sys.path:
    sys.path.append(str(repo_root / 'Code' / 'tools'))

# Import local toolkit (with reload for development)
import local_utility_functions as luf
import geoTERYT_db as gtdb
importlib.reload(gtdb)

print(f"Repo root: {repo_root}")
print(f"GUS root: {gus_root}")
print(f"Geometry root: {geometry_root}")
print(f"geoTERYT_db version: {gtdb.__doc__.split(chr(10))[0] if gtdb.__doc__ else 'unknown'}")

# Check point:
# Load the complete database (built and saved in GUS01C) 
complete_db_path = geospatial_root / 'geoteryt_complete.pkl'
db = gtdb.load_complete_database(complete_db_path)

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Data/GUS
Geometry root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geometry
geoTERYT_db version: 
Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete.pkl...
  Database version: 3.1
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4560 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 0


In [13]:
print(db.get_unit_info("1431011"))

{'teryt_id': '1431011', 'name': 'Warszawa-Bemowo', 'name_dod': 'gmina miejska', 'level': 6, 'kind': 'urban', 'woj': '14', 'pow': '31', 'gmi': '01', 'rodz': '1', 'years_valid': [np.int64(1999), np.int64(2000), np.int64(2001)], 'first_year': np.int64(1999), 'last_year': np.int64(2001), 'past_names': [], 'past_teryt_ids': [], 'past_levels': [], 'past_kinds': [], 'all_teryt_ids': ['1431011'], 'all_names': ['Warszawa-Bemowo'], 'changes': [{'year': np.int64(1999), 'type': 'reform', 'description': 'TYPE_CHANGE: urban -> Warsaw district', 'when_changed': nan}, {'year': np.int64(1999), 'type': 'reform', 'description': 'NAME_CHANGE: Warszawa-Bemowo -> Bemowo', 'when_changed': nan}, {'year': np.int64(1999), 'type': 'reform', 'description': 'DESIGNATION_CHANGE: gmina miejska -> dzielnica', 'when_changed': nan}, {'year': np.int64(1999), 'type': 'reform', 'description': 'POWIAT_CHANGE: 31 -> 65', 'when_changed': nan}, {'year': np.int64(1999), 'type': 'reform', 'description': 'GMINA_CODE_CHANGE: 01 -

## 5. Geometry Operations

In [ ]:
# Convert to GeoDataFrame for a specific year
gdf_1999 = db.to_geodataframe(year=1999, level=6, exclude_subdivisions=True)
gdf_PL = db.get_poland_gdf().boundary

print(f"GeoDataFrame for 1999 gminas: {len(gdf_1999)} rows")
print(f"Columns: {list(gdf_1999.columns)}")
print(f"Rows with geometry: {gdf_1999.geometry.notna().sum()}")


In [ ]:
# Plot gminas with geometry
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
gdf_valid = gdf_1999[gdf_1999.geometry.notna()]
gdf_valid.plot(ax=ax, edgecolor='black', linewidth=0.1, facecolor='lightblue')
#gdf_PL.plot(ax=ax, edgecolor='firebrick', linewidth=2)
ax.set_title(f'Poland Gminas in 1999 ({len(gdf_valid)} with geometry)')
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Merge to voivodeship level
voivodeships_2020 = db.merge_to_level(year=2020, target_level=2)

print(f"Voivodeships (merged): {len(voivodeships_2020)}")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
voivodeships_2020.plot(ax=ax, edgecolor='black', linewidth=1, 
                        column='name', legend=True, legend_kwds={'loc': 'upper left', 'fontsize': 8})
ax.set_title('Poland Voivodeships 2020 (merged from gminas)')
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Merge to powiat level
powiats_2020 = db.merge_to_level(year=2020, target_level=5)

print(f"Powiats (merged): {len(powiats_2020)}")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
powiats_2020.plot(ax=ax, edgecolor='black', linewidth=0.3, facecolor='lightgreen')
ax.set_title(f'Poland Powiats 2020 ({len(powiats_2020)} powiats merged from gminas)')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 6. Compare Years

In [ ]:
# Compare gmina counts across years
years_to_compare = [1999, 2005, 2010, 2015, 2020, 2024]
counts = []

for year in years_to_compare:
    stats = db.get_year_statistics(year)
    gminas = stats['level_counts'].get(6, 0)
    powiats = stats['level_counts'].get(5, 0)
    voivodeships = stats['level_counts'].get(2, 0)
    counts.append({
        'year': year,
        'gminas': gminas,
        'powiats': powiats,
        'voivodeships': voivodeships
    })

counts_df = pd.DataFrame(counts)
print(counts_df.to_string(index=False))

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(counts_df['year'], counts_df['gminas'], color='skyblue')
axes[0].set_title('Gminas per Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')

axes[1].bar(counts_df['year'], counts_df['powiats'], color='lightgreen')
axes[1].set_title('Powiats per Year')
axes[1].set_xlabel('Year')

axes[2].bar(counts_df['year'], counts_df['voivodeships'], color='salmon')
axes[2].set_title('Voivodeships per Year')
axes[2].set_xlabel('Year')

plt.tight_layout()
plt.show()

## 7. Export Database

In [ ]:
# Export to DataFrame
db_df = db.to_dataframe(include_geometry=False)
print(f"Exported {len(db_df)} records to DataFrame")
db_df

In [ ]:
# Save to CSV
output_path = geospatial_root / "geoteryt_database.csv"
db_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"Saved to {output_path}")

In [ ]:
# Export specific year to GeoPackage (if geopandas supports it)
try:
    output_gpkg = geospatial_root / "gminas_2020.gpkg"
    db.export_to_geopackage(output_gpkg, year=2020, level=6)
    print(f"Exported 2020 gminas to {output_gpkg}")
except Exception as e:
    print(f"GeoPackage export not available: {e}")

## 8. Save Complete Database

Save the complete database with all geometries using `save_complete()`. This creates a pickle file that can be loaded with `load_complete_database()` for instant access without rebuilding.

In [ ]:
# Save the complete database (with geometries) using pickle
# If db was created before module reload, attach the new methods

'''
if not hasattr(db, 'save_complete'):
    import types
    db.save_complete = types.MethodType(gtdb.GeoTERYTDatabase.save_complete, db)
    print("ℹ Added save_complete method from reloaded module")

complete_db_path = geospatial_root / "geoteryt_complete.pkl"
db.save_complete(complete_db_path)
print(f"\n✓ Complete database saved to: {complete_db_path}")
'''

## Summary

The GeoTERYT database v3.1 has been successfully built and saved. Key capabilities:

1. **Data Storage**: All administrative divisions from 1999-2024
2. **Historical Tracking**: 
   - Name changes, ID changes, structural changes
   - `historical_codes`: All TERYT codes a unit ever had
   - `code_by_year`: Dict mapping year → teryt_id
3. **Geometry Storage**: Geometry files stored in `_geometries` dict (NOT assigned to records)
4. **Pre-1999 Support**: old_woj, old_woj_id attributes for historical voivodeships
5. **Search**: By TERYT ID, name, year, level, kind
6. **Export**: DataFrame, CSV, GeoPackage, pickle (complete with geometries)
7. **Display**: TERYTRecord.display() for nice Jupyter output, as_df() for DataFrames

### Geometry Assignment (NEW in v3.1)
Geometry assignment methods are demonstrated in **GUS01D_old_division.ipynb**:
- `assign_geometries(year, level)` - exact match assignment
- `assign_missing_geometries(year, level)` - find geometries for unchanged units
- `impute_geometries_past_tid(year, level)` - use code_by_year for affiliated IDs
- `impute_from_best_candidates()` - apply best candidates
- `country_shape_check(year, level)` - verify coverage

### To load the complete database elsewhere:
```python
from geoTERYT_db import load_complete_database
db = load_complete_database("path/to/geoteryt_complete.pkl")
```